In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
from scipy.interpolate import interp1d
import os
from matplotlib.ticker import FormatStrFormatter
import pickle


plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 14


In [ ]:
np.random.seed(99)

In [ ]:
%run ./ESN.ipynb
%run ./Reconstruct.ipynb

window_length = 45


In [ ]:
# load data

with open("./data/hk.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    X = [row for row in reader][1:]
    X = np.array([list(map(float, row[:-1])) for row in X])
    
data_length = X.shape[0]
x = np.linspace(0, data_length - 1, data_length)
int_mul = 15
x_new = np.linspace(0, data_length - 1, data_length * int_mul)

cardio = 0
no2 = 1
so2 = 2
rspar = 3


variables = np.array(['CVDs','$\mathrm{NO}_2$','$\mathrm{SO}_2$','RSP'])
seq = [cardio, no2, so2, rspar]
N_nodes = len(seq)
X_new = np.zeros((data_length * int_mul, X.shape[1]))
i = 0
for item in seq:
    tmp = np.array(X[:,item])
    f = interp1d(x, tmp, kind = 'cubic')
    arr = f(x_new)
    
    X_new[:,i] = arr.T
    i += 1

X = X_new
    
q1 = np.percentile(X, 25, interpolation='midpoint', axis=0)
q2 = np.percentile(X, 50, interpolation='midpoint', axis=0)
q3 = np.percentile(X, 75, interpolation='midpoint', axis=0)

X = (X - q2)/(q3 - q1)

print(X.shape)

N_washout = 10 * int_mul
N_train = 950 * int_mul
N_tstart = 30 * int_mul
N_test = 1000 * int_mul

X_washout = X[:N_washout]

X_t = X[N_washout:N_washout+N_train-1]
Y_t = X[N_washout+1:N_washout+N_train]

X_test = X[N_tstart:N_tstart + N_test]

In [ ]:
Array_to_plot = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]]
labels=['IRC']

N_x = X.shape[1]


N_dim   =  1 * N_x # dimension of inputs (and outputs)


N_net = 30


T, D = X.shape  

k = 0   
TDI_window = []
N_rep, N_evo = 30, 5

offset = np.zeros([N_net, N_rep])
for s in range(N_net):
    for rep in range(N_rep):
        offset[s,rep] = np.random.randint(N_train - N_evo - N_washout  -1)

test_window = 15




X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

for i in range((test_window - 1) * window_length, T):
    segment = X[i - np.arange(test_window) * window_length].reshape(-1)
    X_segments[i - (test_window - 1) * window_length] = segment


N_washout_segments = N_washout
N_train_segments   = N_train
N_tstart_segments  = N_tstart
N_test_segments    = T-N_train_segments


N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
N_units_segments = 150 * N_dim_segments #units in the reservoir 

# washout
X_washout_segments= X_segments[:N_washout_segments]
# training
X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

X_test_segments = X_segments[N_tstart_segments:]

TDInet = np.zeros([N_net,test_window,N_x,N_x])






for net in range(N_net):
    esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
    esn.train(X_washout_segments, X_t_segments, Y_t_segments)

#esn = ESN(N_units, N_dim, N_dim)
#esn.train(X_washout, X_t, Y_t)


    for T in range (0, test_window):
#TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
        TDInet[net, T] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 1, N_dim, N_rep, test_window, 0, 0,  N_washout, N_evo, offset[net], T)
#TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


TDI = np.mean(np.array(TDInet), axis=0)
#TDI_window.append(np.max(TDI)*(softmax_entropy_score(TDI)+ 1e-6))
for t in range (test_window):       
    TDI_window.append(np.max(TDI[t]))
    Array_to_plot [t].append(TDI[t])
#TDI_window.append(np.std(TDI)/(np.mean(TDI) + 1e-6))
#TDI_window.append(np.mean(TDI))
#TDI_window.append(np.max(TDI)/(np.mean(TDI) + 1e-6))
#TDI_window.append(gated_max_softmax_entropy_score(TDI))




plt.plot(TDI_window)
plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])

save_dict = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet,
    'N_x':N_x,
    'test_window':test_window 
    }

save_dir = os.path.join('result', 'CVDs_results')

os.makedirs(save_dir, exist_ok=True) 

file_path = os.path.join(save_dir, 'CVDs_experiment.pkl')


with open(file_path, 'wb') as f:
    pickle.dump(save_dict, f)
    
print(f"data has been saved to: {file_path}")

    



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'CVDs_results', 'CVDs_experiment.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']
N_x = loaded_data['N_x']
test_window = loaded_data['test_window']

In [ ]:
from matplotlib.ticker import FormatStrFormatter
from scipy.interpolate import PchipInterpolator

x = np.arange(len(TDI_window)) * 3
TDI = np.mean(np.array(TDInet), axis=0)
y1 = TDI[:, 3, 0]   # Cardio -> Rspar
y2 = TDI[:, 0, 3]   # Rspar -> Cardio

x_smooth = np.linspace(x.min(), x.max(), 400)

f1 = PchipInterpolator(x, y1)
f2 = PchipInterpolator(x, y2)

y1_smooth = f1(x_smooth)
y2_smooth = f2(x_smooth)

plt.plot(x_smooth, y1_smooth, 'r-', label='Cardio to Rspar')
plt.plot(x_smooth, y2_smooth, 'b-', label='Rspar to Cardio')

plt.plot(x, y1, 'ro', markersize=6)
plt.plot(x, y2, 'bo', markersize=6)

plt.xticks(ticks=x, labels=x)

ax = plt.gca()
y_min, y_max = ax.get_ylim()
new_yticks = np.linspace(y_min, y_max, 4)
plt.yticks(new_yticks)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

filename = f"air_2"
pathname = os.path.join('result', 'figure', f"{filename}.png")
plt.savefig(pathname, bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
x = np.arange(len(TDI_window)) * 3

y1 = TDI[:, 1, 0]   
y2 = TDI[:, 0, 1]   


x_smooth = np.linspace(x.min(), x.max(), 400)

f1 = PchipInterpolator(x, y1)
f2 = PchipInterpolator(x, y2)

y1_smooth = f1(x_smooth)
y2_smooth = f2(x_smooth)

plt.plot(x_smooth, y1_smooth, 'r-', label='Cardio to No2')
plt.plot(x_smooth, y2_smooth, 'b-', label='No2 to Cardio')
plt.plot(x, y1, 'ro', markersize=6)
plt.plot(x, y2, 'bo', markersize=6)

plt.xticks(ticks=x, labels=x)

ax = plt.gca()
y_min, y_max = ax.get_ylim()
new_yticks = np.linspace(y_min, y_max, 4)
plt.yticks(new_yticks)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

filename = f"air_1"
pathname = os.path.join('result', 'figure', f"{filename}.png")
plt.savefig(pathname, bbox_inches='tight', dpi=600)
plt.show()